<a href="https://colab.research.google.com/github/JuliBakagianni/delta_meth/blob/main/run_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sentence_transformers
import transformers
import torch
import sklearn
import yaml

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())

torch version: 2.10.0+cu128
cuda available: True
Device count: 1


In [20]:
#@title git clone
from pathlib import Path
import zipfile, os, shutil
repo_path = '/content/delta_meth'
# rm current folder and clone repo
!rm -rf {repo_path}
print('Cloning repository...')
os.system(f'git clone https://github.com/JuliBakagianni/delta_meth.git {repo_path}')
# interactive upload if segments.zip not present
if not Path('segments.zip').exists():
    try:
        from google.colab import files
        print('Please upload segments.zip (select file in the dialog)')
        uploaded = files.upload()
        for fn, data in uploaded.items():
            open(fn, 'wb').write(data)
    except Exception as e:
        print('Upload skipped or not available:', e)
# ensure target dir exists and extract if zip present
target = Path(repo_path) / 'data' / 'processed' / 'segments'
target.mkdir(parents=True, exist_ok=True)
if Path('segments.zip').exists():
    print('Extracting segments.zip to', target)
    with zipfile.ZipFile('segments.zip', 'r') as z:
        z.extractall(target)
    print('Extraction complete.')

    # Move files one level up if nested 'segments' directory exists
    nested_segments = target / 'segments'
    if nested_segments.exists() and nested_segments.is_dir():
        print('Moving files from nested directory one level up...')
        for item in nested_segments.iterdir():
            dest = target / item.name
            # Remove destination if it already exists to avoid shutil.move errors
            if dest.exists():
                if dest.is_dir():
                    shutil.rmtree(dest)
                else:
                    dest.unlink()
            shutil.move(str(item), str(target))
        nested_segments.rmdir()

    extracted_files = list(target.glob('**/*')) # List all files recursively
    print(f'Found {len(extracted_files)} files after extraction.')
else:
    print('No segments.zip found; ensure JSONs are placed in', target)

Cloning repository...
Extracting segments.zip to /content/delta_meth/data/processed/segments
Extraction complete.
Moving files from nested directory one level up...
Found 291 files after extraction.


### Segmentation evaluation

In [ ]:
import zipfile
import json
import re
from pathlib import Path

# Paths
raw_zip = '/content/evaggelismos_raw_txt_notes.zip'
raw_dir = Path('/content/raw_notes_eval')
seg_dir = Path('/content/delta_meth/data/processed/segments')

# Extract raw notes if needed
if Path(raw_zip).exists() and not raw_dir.exists():
    print(f'Extracting {raw_zip} to {raw_dir}...')
    with zipfile.ZipFile(raw_zip, 'r') as z:
        z.extractall(raw_dir)

# Collect raw and processed files
raw_files = {p.stem: p for p in raw_dir.glob('**/*.txt')} if raw_dir.exists() else {}
processed_files = {p.stem: p for p in seg_dir.glob('**/*.json')}

print(f'Found {len(raw_files)} raw notes and {len(processed_files)} processed JSON files.')

# Evaluation metrics
stats = {
    'total_evaluated': 0,
    'single_segment': 0,
    'multi_segment': 0,
    'multi_segment_correct_start': 0,
    'missing_raw_file': 0,
    'text_length_mismatch': 0,
    'empty_segments': 0
}

# Track anomalies
anomalies = []
empty_segment_notes = []

# Simple regex to guess if dates exist in raw text (adjust based on your actual date format)
date_pattern = re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b')

for stem, json_path in processed_files.items():
    if stem not in raw_files:
        stats['missing_raw_file'] += 1
        continue

    stats['total_evaluated'] += 1
    raw_text = raw_files[stem].read_text(encoding='utf-8')

    try:
        data = json.loads(json_path.read_text(encoding='utf-8'))
        segs = data.get('segments', [])
    except Exception as e:
        print(f'Error reading {json_path}: {e}')
        continue

    seg_len = len(segs)
    total_seg_text_len = 0

    for idx, s in enumerate(segs):
        text = s.get('text', '')
        total_seg_text_len += len(text)
        if not text.strip():
            stats['empty_segments'] += 1
            empty_segment_notes.append({'stem': stem, 'date': s.get('date', 'Unknown'), 'index': idx})

    raw_text_len = len(raw_text)

    # Check length mismatch (allowing 10% tolerance for headers/whitespace removal)
    if raw_text_len > 0 and (total_seg_text_len / raw_text_len < 0.90 or total_seg_text_len / raw_text_len > 1.10):
        stats['text_length_mismatch'] += 1

    if seg_len == 1:
        stats['single_segment'] += 1
        # Optional: check if raw_text actually contains dates. If it does, maybe it should have been segmented.
    elif seg_len > 1:
        stats['multi_segment'] += 1
        # Check if first segment is 'pre ICU' / 'before_ICU'
        first_date = str(segs[0].get('date', '')).lower()
        if 'before' in first_date or 'pre' in first_date:
            stats['multi_segment_correct_start'] += 1
        else:
            anomalies.append({'stem': stem, 'first_date': segs[0].get('date', 'Unknown')})

print('\n--- Segmentation Evaluation Results ---')
for k, v in stats.items():
    print(f'{k}: {v}')

if stats['multi_segment'] > 0:
    correct_start_pct = (stats['multi_segment_correct_start'] / stats['multi_segment']) * 100
    print(f'\nMulti-segment notes correctly starting with pre-ICU: {correct_start_pct:.1f}%')

    if empty_segment_notes:
        print(f'\n--- Found {len(empty_segment_notes)} empty segments ---')
        for a in empty_segment_notes[:15]:
            print(f"Note ID: {a['stem']} -> Segment Index: {a['index']}, Date: '{a['date']}'")
        if len(empty_segment_notes) > 15:
            print(f"... and {len(empty_segment_notes) - 15} more.")
    else:
        print('\n--- Good news: No empty segments found! ---')

    if anomalies:
        print(f'\n--- Investigating {len(anomalies)} multi-segment notes without pre-ICU start ---')
        for a in anomalies[:15]: # Show up to 15 examples
            print(f"Note ID: {a['stem']} -> Starts with Date: '{a['first_date']}'")
        if len(anomalies) > 15:
            print(f"... and {len(anomalies) - 15} more.")


Found 574 raw notes and 287 processed JSON files.

--- Segmentation Evaluation Results ---
total_evaluated: 287
single_segment: 131
multi_segment: 156
multi_segment_correct_start: 137
missing_raw_file: 0
text_length_mismatch: 1
empty_segments: 5

Multi-segment notes correctly starting with pre-ICU: 87.8%

--- Found 5 empty segments ---
Note ID: 25363214 -> Segment Index: 1, Date: '2024-06-03'
Note ID: 21363910 -> Segment Index: 29, Date: '2024-05-13'
Note ID: 25426210 -> Segment Index: 4, Date: '2024-09-09'
Note ID: 25477377 -> Segment Index: 12, Date: '26/11'
Note ID: 25477377 -> Segment Index: 13, Date: '27/11'

--- Investigating 19 multi-segment notes without pre-ICU start ---
Note ID: 23458098 -> Starts with Date: '22/11'
Note ID: 23458098Γò¼ΓûÆ -> Starts with Date: '25/10-21'
Note ID: 25332587 -> Starts with Date: '2024-04-21'
Note ID: 24954092 -> Starts with Date: '02-17/07'
Note ID: 20097899 -> Starts with Date: '11/03'
Note ID: 25349980 -> Starts with Date: '2024-05-22'
Note ID

---

### Call pipeline functions interactively

This cell imports `run_pipeline` and runs sequential comparisons for a single note's segments so you can inspect intermediate outputs. If model loading fails, a simple mock fallback will be used for demonstration.

In [6]:
import sys, json, glob
from pathlib import Path
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.run_pipeline import run_pipeline


# list available segment JSONs
seg_dir = Path(repo_path) / 'data' / 'processed' / 'segments'
files = sorted([p for p in seg_dir.glob('**/*.json')]) if seg_dir.exists() else [] # Modified to look for JSONs recursively
print('Found', len(files), 'segment files')
# pick first multi-segment file for interactive inspection
sample = None
for p in files:
    data = json.loads(p.read_text(encoding='utf-8'))
    if len(data.get('segments', [])) > 1:
        sample = p
        break
if not sample:
    print('No multi-segment file found; add JSONs to', seg_dir)
else:
    print('Using sample file:', sample)
    note = json.loads(sample.read_text(encoding='utf-8'))
    segs = note.get('segments', [])
    print('Segment count:', len(segs))
    # sequentially compare segment[i] vs segment[i+1]
    for i in range(len(segs)-1):
        a = segs[i].get('text','')
        b = segs[i+1].get('text','')
        print('Comparison', i, 'dates:', segs[i].get('date'), 'vs', segs[i+1].get('date'))
        try:
            contradiction, detailed = run_pipeline(note_a=a, note_b=b, config_path=str(Path(repo_path)/'configs'/'config.yaml'), verbose=True, chunk_unit='sentences', chunk_size=1)
            print('Contradiction:', contradiction)
        except Exception as e:
            print('Real pipeline failed:', e)
            # Mock fallback: simple heuristic demo
            from src.preprocessing.chunking import chunk_notes
            ca = chunk_notes(a, chunk_size=1, chunk_unit='sentences')
            cb = chunk_notes(b, chunk_size=1, chunk_unit='sentences')
            detailed = []
            for idx, sent in enumerate(ca):
                other = cb[idx] if idx < len(cb) else (cb[-1] if cb else '')
                label = 'contradiction' if ('πυρετό' in sent and 'πυρετό' not in other) or ('πυρετό' in other and 'πυρετό' not in sent) else 'neutral'
                detailed.append({'i': idx, 'j': idx if idx < len(cb) else len(cb)-1, 'nli_label': label, 'chunk1': sent, 'chunk2': other})
            contradictions = [d for d in detailed if d['nli_label']=='contradiction']
            contradiction = contradictions[0] if contradictions else None
            print('Mock contradiction:', contradiction)

Found 287 segment files
Using sample file: /content/delta_meth/data/processed/segments/segments/20052347.json
Segment count: 3
Comparison 0 dates: before_ICU vs 2024-01-23
[pipeline] note A -> 6 chunks
[pipeline] note B -> 5 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pipeline] aligned candidate pairs (sim>=0.5): 4
[pipeline] pair (0,6) sim=0.560 nli=entailment (0.995)
  A: Η ασθενής μεταφέρεται στη ΜΕΘ μετά τη θρομβεκτομή για παρακολούθηση
  B: Mεταφέρεται στη νευρολογική κλινική για συνέχιση νοσηλείας
[pipeline] pair (1,0) sim=0.578 nli=entailment (0.914)
  A: Σε αυτόματη αναπνοή σε VM50% με καλή αεριομετρική εικόνα
  B: Ιδιες συνθήκες αναπνευστικά
[pipeline] pair (2,3) sim=0.540 nli=entailment (0.512)
  A: Αιμοδυναμικά σταθερή
  B: Διούρηση ικανοποιητική
[pipeline] pair (4,1) sim=0.736 nli=contradiction (0.993)
  A: GCS: 12/15 (δεν ανοίγει τους οφθαλμούς), ΑΡ πυραμιδική συνδρομή- εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί
  B: GCS: 14/15 (ανοίγει τους οφθαλμούς στα παραγγέλματα, εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί)
[pipeline] contradiction FOUND:
  sim_score=0.736
  nli_confidence=0.993
  chunk A:
    GCS: 12/15 (δεν ανοίγει τους οφθαλμούς), ΑΡ πυραμιδική συνδρομή- εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί
  chunk B:
    GCS: 14/15 (ανοίγει τους οφθαλμούς στ

---

### Run Segmentation+Similarity+NLI filtering


In [13]:
#@title get a sample
import random

# Sampling parameters
SAMPLE_SIZE = 10   # set None to process all multi-segment files
SAMPLE_SEED = 42   # reproducible seed for sampling

# All segment JSONs
all_files = sorted([p for p in seg_dir.glob('**/*.json')])

# Keep only multi-segment files (>1 segment)
multi_files = []
for p in all_files:
    try:
        note = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    if len(note.get('segments', [])) > 1:
        multi_files.append(p)

# Select sample (reproducible)
if SAMPLE_SIZE is None or SAMPLE_SIZE >= len(multi_files):
    files = multi_files
else:
    rng = random.Random(SAMPLE_SEED)
    files = rng.sample(multi_files, SAMPLE_SIZE)

# Optionally apply PROCESS_LIMIT on top of sampled files (for tiny quick runs)
if 'PROCESS_LIMIT' in globals() and PROCESS_LIMIT:
    files = files[:PROCESS_LIMIT]

In [15]:
#@title multilingual nli
from pathlib import Path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.batch_runner import run_batch

OUT_DIR = '/content/delta_meth/data/results/diagnostic_shifts'
LOCAL_CSV = str(Path(OUT_DIR) / 'threshold_logs.csv')  # will live inside the auto-subdir

run_batch(
    seg_dir='/content/delta_meth/data/processed/segments',
    out_dir=OUT_DIR,
    local_csv=LOCAL_CSV,
    drive_root='/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli',
    auto_subdir=True,
    sample_size=10,
    sample_seed=42,
    config_path=str(Path('/content/delta_meth/configs/config.yaml')),
    translate_nli=False
)

Processing notes:   0%|          | 0/10 [00:00<?, ?it/s]

[pipeline] note A -> 10 chunks
[pipeline] note B -> 6 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pipeline] aligned candidate pairs (sim>=0.5): 1
[pipeline] contradiction FOUND:
  sim_score=0.718
  nli_confidence=0.956
  chunk A:
    Αιμοδυναμικά με υπερτασικό προφίλ υπό λαβηταλόλη ενδοφλεβίως
  chunk B:
    Αιμοδυναμικά σταθερός, προοδευτικός απογαλακτισμός από τη λαβηταλόλη
Run complete


In [8]:
#@title greek nli
from pathlib import Path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.batch_runner import run_batch

OUT_DIR = '/content/delta_meth/data/results/diagnostic_shifts'
LOCAL_CSV = str(Path(OUT_DIR) / 'threshold_logs.csv')  # will live inside the auto-subdir

run_batch(
  seg_dir='/content/delta_meth/data/processed/segments',
  out_dir=OUT_DIR,
  local_csv=LOCAL_CSV,
  drive_root='/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli',
  config_path=str(Path(repo_path) / 'configs' / 'config.yaml'),
  auto_subdir=True,
  sample_size=10,
  sample_seed=42,
  nli_model='lighteternal/nli-xlm-r-greek',
  translate_nli=False
)

auto dir= True, dir = /content/delta_meth/data/results/diagnostic_shifts/lighteternal-nli-xlm-r-greek_seed42
Running batch on 10 files (sample_size=10, process_limit=None)...
len filtered = 4 
[pipeline] note A -> 10 chunks
[pipeline] note B -> 6 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: lighteternal/nli-xlm-r-greek
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

[pipeline] contradiction FOUND:
  sim_score=0.655
  nli_confidence=0.935
  chunk A:
    Αιμοδυναμικά σε διπλή αγγειοσύσπαση με μειούμενες δόσεις ωστόσο
  chunk B:
    Παραμένει ουδετεροπενική και θρομβοπενική με αναγκη καθημερινων μεταγγίσεων με αιμοπεταλια και χορήγηση αυξητικού παράγοντα
[pipeline] note A -> 6 chunks
[pipeline] note B -> 7 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 8
[pipeline] contradiction FOUND:
  sim_score=0.624
  nli_confidence=0.993
  chunk A:
    Αιμοδυναμικά σε διπλή αγγειοσύσπαση με διακοπή της βαζοπρεσσίνης σταδιακά στις 14/02 και διακοπή νοραδρεναλίνης στις 15/02
  chunk B:
    Υπό χαμηλή δόση ρεμιφαιντανύλης, χωρίς αγγειοσυσπαστικά
[pipeline] note A -> 7 chunks
[pipeline] note B -> 8 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 3
[pipeline] no contradiction found (above threshold)
len filtered = 4 
[pipeline] note A -> 13 chunks
[pipeline] note B -> 11 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 11
[pipeline] contradiction FO

In [10]:
#@title translation and english nli

from pathlib import Path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.batch_runner import run_batch

OUT_DIR = '/content/delta_meth/data/results/diagnostic_shifts'
LOCAL_CSV = str(Path(OUT_DIR) / 'threshold_logs.csv')  # will live inside the auto-subdir

run_batch(
  seg_dir='/content/delta_meth/data/processed/segments',
  out_dir=OUT_DIR,
  local_csv=LOCAL_CSV,
  drive_root='/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli',  # or None
  sample_size=10,
  sample_seed=42,
  auto_subdir=True,
  config_path=str(Path(repo_path) / 'configs' / 'config.yaml'),
  nli_model='facebook/bart-large-mnli',            # English NLI model
  translate_nli=True,
  translation_model='facebook/nllb-200-distilled-1.3B'   # Greek->English translator (example)
)


[pipeline] note A -> 10 chunks
[pipeline] note B -> 6 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 10


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

[pipeline] no contradiction found (above threshold)
[pipeline] note A -> 6 chunks
[pipeline] note B -> 7 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 8
[pipeline] contradiction FOUND:
  sim_score=0.575
  nli_confidence=0.877
  chunk A:
    Αιμοδυναμικά σε διπλή αγγειοσύσπαση με διακοπή της βαζοπρεσσίνης σταδιακά στις 14/02 και διακοπή νοραδρεναλίνης στις 15/02
  chunk B:
    ΄Ελαβε 1 ΜΣΕ 16/02 και 1 pool αιμοπε΄ταλίων 17/01
[pipeline] note A -> 7 chunks
[pipeline] note B -> 8 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 3
[pipeline] no contradiction found (above threshold)
[pipeline] note A -> 13 chunks
[pipeline] note B -> 11 chunks
[pipeline] aligned candidate pairs (sim>=0.5): 11
[pipeline] contradiction FOUND:
  sim_score=0.609
  nli_confidence=0.886
  chunk A:
    Εκτίμηση αεραγωγού από ΩΡΛ, αιθάλη έως την γλωττίδα, διασωλήνωση για προστασία αεραγωγού και 500mg Solucortef iv άπαξ
  chunk B:
    Συνεστήθη χορήγηση δεξαμεθαζόνης 8mg/d για δύο ημέρες κι επανεκτίμηση

[PosixPath('/content/delta_meth/data/processed/segments/20984634.json'),
 PosixPath('/content/delta_meth/data/processed/segments/20144314.json'),
 PosixPath('/content/delta_meth/data/processed/segments/24560882.json'),
 PosixPath('/content/delta_meth/data/processed/segments/23470427.json'),
 PosixPath('/content/delta_meth/data/processed/segments/22650469.json'),
 PosixPath('/content/delta_meth/data/processed/segments/21192057.json'),
 PosixPath('/content/delta_meth/data/processed/segments/20838717.json'),
 PosixPath('/content/delta_meth/data/processed/segments/25435079.json'),
 PosixPath('/content/delta_meth/data/processed/segments/20755954.json'),
 PosixPath('/content/delta_meth/data/processed/segments/25494828.json')]

In [18]:
#@title export contradiction pairs for LLM labeling (CSV)
from pathlib import Path
import sys
import importlib

repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Force Python to clear the cached module so it reads the freshly cloned version
if 'src.pipeline.batch_runner' in sys.modules:
    del sys.modules['src.pipeline.batch_runner']

from src.pipeline.batch_runner import run_batch
from src.pipeline.batch_runner import export_llm_candidates_csv

# Root that contains your setting folders (e.g. delta_meth_multi_nli, delta_meth_greek_nli, etc.)
RESULTS_ROOT = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/diagnostic_shifts'

OUT_CSV = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/llm_candidates_all_settings.csv'

summary = export_llm_candidates_csv(
    results_dir=RESULTS_ROOT,
    out_csv=OUT_CSV,
)

print('CSV ready:', OUT_CSV)
print(summary)

{
  "results_dir": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/diagnostic_shifts",
  "out_csv": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/llm_candidates_all_settings.csv",
  "json_files_found": 10,
  "note_files_read": 10,
  "bad_files": 0,
  "candidate_rows": 35
}
CSV ready: /content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/llm_candidates_all_settings.csv
{'results_dir': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/diagnostic_shifts', 'out_csv': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/llm_candidates_all_settings.csv', 'json_files_found': 10, 'note_files_read': 10, 'bad_files': 0, 'candidate_rows': 35}


In [19]:
#@title export contradiction pairs for LLM labeling (CSV)
from pathlib import Path
import sys
import importlib

repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if 'src.pipeline.batch_runner' in sys.modules:
    del sys.modules['src.pipeline.batch_runner']

from src.pipeline.batch_runner import export_llm_candidates_csv

# Root that contains your setting folders (e.g. delta_meth_multi_nli, delta_meth_greek_nli, etc.)
RESULTS_ROOT = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/diagnostic_shifts'

OUT_CSV = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/llm_candidates_all_settings.csv'

summary = export_llm_candidates_csv(
    results_dir=RESULTS_ROOT,
    out_csv=OUT_CSV,
)

print('CSV ready:', OUT_CSV)
print(summary)

# Root that contains your setting folders (e.g. delta_meth_multi_nli, delta_meth_greek_nli, etc.)
RESULTS_ROOT = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/diagnostic_shifts'

OUT_CSV = '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_candidates_all_settings.csv'

summary = export_llm_candidates_csv(
    results_dir=RESULTS_ROOT,
    out_csv=OUT_CSV,
)

print('CSV ready:', OUT_CSV)
print(summary)

{
  "results_dir": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/diagnostic_shifts",
  "out_csv": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/llm_candidates_all_settings.csv",
  "json_files_found": 10,
  "note_files_read": 10,
  "bad_files": 0,
  "candidate_rows": 22
}
CSV ready: /content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/llm_candidates_all_settings.csv
{'results_dir': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/diagnostic_shifts', 'out_csv': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/llm_candidates_all_settings.csv', 'json_files_found': 10, 'note_files_read': 10, 'bad_files': 0, 'candidate_rows': 22}
{
  "results_dir": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/diagnostic_shifts",
  "out_csv": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_candidates_all_settings.csv",
  "json_files_found": 11,
  "note_files_read": 11,
  "bad_f

In [23]:
#@title LLM diagnostic-shift labeling (Llama3-70B on Bedrock)
from pathlib import Path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if 'src.pipeline.batch_runner' in sys.modules:
    del sys.modules['src.pipeline.batch_runner']

# If needed in Colab:
!pip -q install boto3

from src.pipeline.batch_runner import classify_candidates_with_llm

SEG_DIR = '/content/delta_meth/data/processed/segments'
AWS_JSON = '/content/aws.json'

nli_annotations_csv_list = [
    '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_candidates_all_settings.csv',
    '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_transl_nli/llm_candidates_all_settings.csv',
    '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_greek_nli/llm_candidates_all_settings.csv'
]
for cand_csv in nli_annotations_csv_list:
  out_csv = cand_csv.replace('llm_candidates_all_settings.csv', 'llm_annotator_ready.csv')
  summary = classify_candidates_with_llm(
      candidates_csv=cand_csv,
      seg_dir=SEG_DIR,
      aws_json_path=AWS_JSON,
      out_csv=out_csv,
      model_id='meta.llama3-70b-instruct-v1:0',
      temperature=0.0,
      max_tokens=300,
      sleep_seconds=0.0,
  )

  print(summary)
  print('Saved:', out_csv)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00
{
  "candidates_csv": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_candidates_all_settings.csv",
  "seg_dir": "/content/delta_meth/data/processed/segments",
  "out_csv": "/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_annotator_ready.csv",
  "model_id": "meta.llama3-70b-instruct-v1:0",
  "temperature": 0.0,
  "rows_total": 28,
  "rows_with_parsed_json": 28,
  "llm_errors": 0,
  "parse_errors": 0,
  "missing_segment_context_rows": 0
}
{'candidates_csv': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_candidates_all_settings.csv', 'seg_dir': '/content/delta_meth/data/processed/segments', 'out_csv': '/content/drive/MyDrive/ICU_diagnostic_shift/delta_meth_multi_nli/llm_annotator_ready.csv